*This code is a companion to the book **Mastering PyTorch and Lightning: A Step-by-Step Practical Guide with QA** by Aghiles Kebaili*
> This notebook contains the raw code for Chapter 1: The Anatomy of Tensors. To understand the tensor model behind the rest of the book and the practical debugging mindset behind each example, get the full step-by-step guide on Amazon: **[Get the Book Here](https://www.amazon.fr/Mastering-PyTorch-Lightning-Step-Step-ebook/dp/B0HGNZS55V)**

At the heart of PyTorch sits the tensor: a structured view over memory with shape, stride, dtype, and device metadata. If you understand these properties well, many debugging and performance questions stop feeling mysterious.

The first section is intentionally practical. We inspect how a tensor stores its data, how copies differ from shared views, and why the same values can behave very differently depending on dtype and layout.

## 1. Internal Architecture & Memory Allocation
### Step 1: Tensor Header and Storage

In [ ]:
import torch

# A PyTorch tensor combines a one-dimensional storage buffer with metadata:
# shape (logical dimensions), stride (how to walk through storage), dtype, and device.
# This separation is the key to understanding tensor behavior.
matrix = torch.arange(6, dtype=torch.float32).reshape(2, 3)
print('Shape:', matrix.shape)   
print('Stride:', matrix.stride()) # (3, 1)
print('Storage offset:', matrix.storage_offset()) # 0
print('Dtype:', matrix.dtype) # torch.float32

print('Device:', matrix.device) # cpu### Step 2: Distinguish Copies from Shared Memory

print('Allocated data bytes:', matrix.numel() * matrix.element_size()) # 24

Shape: torch.Size([2, 3])
Stride: (3, 1)
Storage offset: 0
Dtype: torch.float32
Device: cpu
Allocated data bytes: 24


### Step 2: Distinguish Copies from Shared Memory

In [ ]:
import numpy as np
import torch

# PyTorch provides several creation functions with different memory behaviors.
# torch.from_numpy() reuses existing NumPy memory (zero-copy).
# torch.tensor() allocates a new independent buffer and copies the data.
np_array = np.array([1.0, 2.0, 3.0], dtype=np.float32)

# Memory sharing: modifying t_shared modifies np_array
t_shared = torch.from_numpy(np_array)
# Alternative: torch.as_tensor(np_array)

# Explicit copy: allocation of a new independent Storage
t_copy = torch.tensor(np_array)

# Verifying memory sharing
t_shared[0] = 99.0
print(f"NumPy array after mutation: {np_array[0]}") # 99.0
print(f"Copied tensor (independent): {t_copy[0]}") # 1.0
assert np_array[0] == t_shared[0].item()
assert t_copy[0].item() == 1.0

NumPy array after mutation: 99.0
Copied tensor (independent): 1.0


### Step 3: Control dtype and Memory Footprint

In [ ]:
import torch

# Choosing dtype affects memory footprint and numerical behavior.
# Lower precision reduces storage but changes rounding and range properties.
weights_fp32 = torch.empty((1024, 1024), dtype=torch.float32)
weights_bf16 = torch.empty((1024, 1024), dtype=torch.bfloat16)

fp32_bytes = weights_fp32.numel() * weights_fp32.element_size()
bf16_bytes = weights_bf16.numel() * weights_bf16.element_size()
print(fp32_bytes, bf16_bytes)
assert bf16_bytes == fp32_bytes // 2

# Cast explicitly when an operation requires another dtype.
x = torch.arange(10, dtype=torch.int64)
x_float = x.to(dtype=torch.float32)
assert x_float.dtype == torch.float32

4194304 2097152


A tensor may look like a simple container, but moving it between CPU and GPU is one of the most common production bottlenecks. This is why host memory, pinned transfers, and device placement matter so much in real training pipelines.

### Step 4: Move Tensors Between Host and Accelerator

In [ ]:
import torch

# Host-to-device transfer is a major bottleneck in training.
# Pinned memory allows the GPU to perform asynchronous copies when non_blocking=True.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tensor = torch.zeros((4, 512, 512), dtype=torch.float32)

if device.type == "cuda":
    host_tensor = tensor.pin_memory()  # Lock in RAM to enable efficient GPU transfer
    device_tensor = host_tensor.to(device, non_blocking=True)  # Asynchronous copy
else:
    device_tensor = tensor.to(device)


print(device_tensor.device)
assert device_tensor.shape == tensor.shape

cuda:0


## 2. Advanced Dimension & Stride Manipulation
### Step 1: Read Strides and Calculate an Offset

In [ ]:
# Strides determine how to traverse a flat storage buffer.
# For a matrix, stride (3, 1) means: advance 3 elements to move one row, 1 element to move one column.
# This is how a linear buffer is interpreted as a multidimensional tensor.
x = torch.arange(6, dtype=torch.int32).reshape(2, 3)

print("Shape:", x.shape)                 # torch.Size([2, 3])
print("Strides:", x.stride())            # (3, 1)
print("Contiguous:", x.is_contiguous())  # True

row, column = 1, 2
# The flat offset is computed from the base storage plus row/column travel.
offset = (
    x.storage_offset()
    + row * x.stride(0)
    + column * x.stride(1)
)
print("Calculated offset:", offset)       # 5
assert x[row, column] == x.reshape(-1)[offset]

Shape: torch.Size([2, 3])
Strides: (3, 1)
Contiguous: True
Calculated offset: 5


### Step 2: Compare Contiguous and Channels-Last Layouts

In [ ]:
import torch

# The standard C-contiguous layout (NCHW) is not always optimal for computation.
# Channels Last can improve performance on compatible hardware by changing physical strides.
# Logical shape remains (B, C, H, W), but memory layout is reordered.
vision_tensor = torch.randn(32, 3, 224, 224)
print("Is C-contiguous:", vision_tensor.is_contiguous())  # True

# Convert to Channels Last for optimized Conv2D kernels on compatible hardware
channels_last_tensor = vision_tensor.to(memory_format=torch.channels_last)

# Logical shape remains (B, C, H, W), but physical strides change.
print("Is C-contiguous:", channels_last_tensor.is_contiguous())  # False
print(
    "Is Channels-Last contiguous:",
    channels_last_tensor.is_contiguous(memory_format=torch.channels_last),
)  # True

assert channels_last_tensor.shape == vision_tensor.shape
assert channels_last_tensor.stride() != vision_tensor.stride()

Is C-contiguous: True
Is C-contiguous: False
Is Channels-Last contiguous: True


### Step 3: Choose Between view(), reshape(), and contiguous()

In [ ]:
import torch

# Transpose and view are both operations on tensor metadata.
# A transpose is a view that only changes strides, not storage.
# An incompatible view() may fail if the new shape doesn't match the stride pattern.
A = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)

# Transposition changes the strides without moving a byte
A_t = A.t()
print("A_t shape:", A_t.shape)       # (3, 2)
print("A_t strides:", A_t.stride())  # (1, 3), non-contiguous

# Attempting an incompatible view raises RuntimeError
# view() requires a compatible layout, while transpose changes the stride pattern.
try:
    A_t.view(-1)
except RuntimeError as error:
    print(f"Error caught :\n{error}")

# Solution 1: force contiguity before viewing
flat_view = A_t.contiguous().view(-1)

# Solution 2: reshape handles the copy automatically when needed
flat_reshaped = A_t.reshape(-1)

assert torch.equal(flat_view, flat_reshaped)

A_t shape: torch.Size([3, 2])
A_t strides: (1, 3)
Error caught :
view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.


### Step 4: Reorder Tensor Dimensions

In [ ]:
images = torch.randn(32, 3, 224, 160)

# transpose() swaps two dimensions; permute() rearranges all dimensions.
# Both are views that change strides without moving bytes.
swapped_hw = images.transpose(2, 3)  # Swap height and width
print(swapped_hw.shape)  # (32, 3, 160, 224)

nhwc = images.permute(0, 2, 3, 1)  # Rearrange from NCHW to NHWC
print(nhwc.shape)  # (32, 224, 160, 3)

torch.Size([32, 3, 160, 224])
torch.Size([32, 224, 160, 3])


### Step 5: Add and Remove Singleton Dimensions

In [ ]:
feature_map = torch.randn(64, 128)  # (Batch, Features)

# unsqueeze() inserts a singleton dimension (size=1).
# squeeze() removes singleton dimensions. Both are views with no data movement.
expanded_1 = feature_map.unsqueeze(1)  # Insert dimension at position 1: (64, 1, 128)

# Idiomatic injection via None / np.newaxis
expanded_2 = feature_map[:, None, :]

# Remove the singleton dimension
collapsed = expanded_1.squeeze(dim=1)  # Returns to (64, 128)

assert expanded_1.shape == expanded_2.shape and torch.equal(collapsed, feature_map)

print("Expanded shape:", expanded_1.shape)  # (64, 1, 128)
print("Collapsed shape:", collapsed.shape)  # (64, 128)

Expanded shape: torch.Size([64, 1, 128])
Collapsed shape: torch.Size([64, 128])


### Step 6: Flatten and Restore Structured Features

In [ ]:
import torch

# Convolutional features output spatial structure that Linear layers cannot process directly.
# flatten() merges dimensions; unflatten() restores them when the original structure is known.
conv_out = torch.randn(16, 64, 7, 7)  # (Batch, Channels, Height, Width)

# Flatten for ingestion into a Linear layer: (Batch, Channels * Height * Width)
flat = torch.flatten(conv_out, start_dim=1)  # (16, 3136)

# Inverse structured unfolding
unflat = flat.unflatten(dim=1, sizes=(64, 7, 7))  # (16, 64, 7, 7)

assert torch.equal(unflat, conv_out)

print("Flat shape:", flat.shape)      # (16, 3136)
print("Unflat shape:", unflat.shape)  # (16, 64, 7, 7)

Flat shape: torch.Size([16, 3136])
Unflat shape: torch.Size([16, 64, 7, 7])


### Gotcha 1: Graph Accumulation and VRAM Memory Leaks

In [ ]:
# During training, accumulating loss tensors keeps their entire computation graph in memory.
# This includes intermediate activations saved for backward pass, which can cause VRAM leaks.
# Using .item() to extract scalars breaks the graph and prevents runaway memory use.
def count_graph_nodes(tensor):
    """Count unique Autograd nodes reachable from a tensor."""
    if not isinstance(tensor, torch.Tensor) or tensor.grad_fn is None:
        return 0
        
    visited_nodes = set()
    
    def traverse(grad_fn):
        if grad_fn is None or grad_fn in visited_nodes:
            return
        visited_nodes.add(grad_fn)
        
        # Traverse the computation graph by following next_functions
        for next_fn, _ in getattr(grad_fn, 'next_functions', []):
            traverse(next_fn)
            
    traverse(tensor.grad_fn)
    return len(visited_nodes)

In [ ]:
import torch

# Metric accumulation is a common source of VRAM leaks in training loops.
# Tensors retain their grad_fn, keeping graphs alive; scalars do not.
graph_total = 0.0
dataloader = [(torch.randn(4, 3, 32, 32), torch.randint(0, 2, (4,))) for _ in range(5)]
model = torch.nn.Linear(3 * 32 * 32, 2)  # Example model
criterion = torch.nn.CrossEntropyLoss()

# BAD: tensor accumulation retains every intermediate graph from every batch.
for data, target in dataloader:
    output = model(data.flatten(start_dim=1))
    loss = criterion(output, target)
    graph_total = graph_total + loss  # Accumulating tensors = accumulating graphs

# GOOD: .item() extracts a plain Python scalar with no Autograd history.
scalar_total = 0.0
for data, target in dataloader:
    output = model(data.flatten(start_dim=1))
    loss = criterion(output, target)
    scalar_total += loss.item()  # Plain floats cost no VRAM

# display the number of nodes in the computation graph
print("Number of nodes in the computation graph:", count_graph_nodes(graph_total))
print("Number of nodes in the scalar total computation graph:", count_graph_nodes(torch.tensor(scalar_total)))

Number of nodes in the computation graph: 27
Number of nodes in the scalar total computation graph: 0


### Gotcha 2: In-Place Operations and Graph Breaking

In [ ]:
# In-place operations (ending with _) modify existing tensors.
# They can corrupt gradients if Autograd saved that tensor for backward pass.
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * 2
objective = y.pow(2).sum()  # Backward() will need the original value of y

# This in-place mutation invalidates the saved y before backward() can use it.
y.add_(1.0)  # Modifies y before backward has finished

try:
    objective.backward()
except RuntimeError as error:
    print(f"Autograd detected the in-place modification: {error}")

Autograd detected the in-place modification: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.FloatTensor [3]], which is output 0 of AddBackward0, is at version 1; expected version 0 instead. Hint: enable anomaly detection to find the operation that failed to compute its gradient, with torch.autograd.set_detect_anomaly(True).


### Gotcha 3: Implicit Typing from NumPy to PyTorch

In [ ]:
import numpy as np
import torch

# NumPy defaults to float64; PyTorch models often expect float32.
# torch.from_numpy() preserves the NumPy dtype, so conversion is needed.
f32_model = torch.nn.Linear(10, 5)  # Model parameters are torch.float32 by default

raw_data = np.random.randn(10, 10)        # NumPy defaults to float64
tensor_data = torch.from_numpy(raw_data)  # Preserves dtype: torch.float64

try:
    forward_f32 = f32_model(tensor_data)  # This will raise a RuntimeError due to dtype mismatch
except RuntimeError as error:
    print(f"Error occurred: {error}")

# Explicitly cast before passing the tensor into the model
# This is a standard fix when bringing data from NumPy into a PyTorch model.
tensor_data = tensor_data.to(torch.float32)
forward_f32 = f32_model(tensor_data)  # This will now work correctly
assert forward_f32.dtype == torch.float32

print("Forward pass successful with dtype:", forward_f32.dtype)  # torch.float32

Error occurred: mat1 and mat2 must have the same dtype, but got Double and Float
Forward pass successful with dtype: torch.float32
